In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


EMB_PATH = Path("../data/processed/08.FRAIL_with_tsvd_10dim.csv")
LLM_PATH = Path("../data/processed/09.llm.csv")

emb = pd.read_csv(EMB_PATH)
llm = pd.read_csv(LLM_PATH)

merge_keys = [k for k in ["ID_Aluno", "Momento"] if k in emb.columns and k in llm.columns]
if not merge_keys:
    merge_keys = ["ID_Aluno"]

df = emb.merge(llm, on=merge_keys, how="left", suffixes=("", "_llm"))

df = df.rename(columns={c: c.replace("tentaiva", "tentativa") for c in df.columns if "tentaiva" in c})

right_attempts = [c for c in df.columns if re.fullmatch(r"HANDGRIP_DIREITA_tentativa\d+", c)]
left_attempts  = [c for c in df.columns if re.fullmatch(r"HANDGRIP_ESQUERDA_tentativa\d+", c)]

def side_max(attempt_cols, fallback_mean_col):
    if attempt_cols:
        tmp = df[attempt_cols].apply(pd.to_numeric, errors="coerce")
        return tmp.max(axis=1, skipna=True)
    if fallback_mean_col in df.columns:
        return pd.to_numeric(df[fallback_mean_col], errors="coerce")
    return pd.Series(np.nan, index=df.index)

df["HANDGRIP_BEST"] = np.nanmax(
    np.column_stack([
        side_max(right_attempts, "HANDGRIP_DIREITA_mean"),
        side_max(left_attempts,  "HANDGRIP_ESQUERDA_mean"),
    ]),
    axis=1
)

# -------------------------
# 3) Helpers: seleção + erros úteis
# -------------------------
def pick_col(df, preferred, keywords_all):
    if preferred in df.columns:
        return preferred
    cands = [c for c in df.columns if all(k.lower() in c.lower() for k in keywords_all)]
    return cands[0] if cands else None

def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def candidate_cols(df, keywords_any, max_show=15):
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[:max_show]

def assert_mapped(var_name, col_value, df, suggest_keywords):
    if col_value is None:
        cands = candidate_cols(df, suggest_keywords, max_show=20)
        raise ValueError(
            f"[MAPPING ERROR] '{var_name}' não foi encontrado.\n"
            f"  - Procurei por keywords: {suggest_keywords}\n"
            f"  - Candidatos no df.columns (top {len(cands)}): {cands}\n"
        )

def assert_exists(var_name, col_value, df):
    if col_value not in df.columns:
        raise ValueError(
            f"[MAPPING ERROR] '{var_name}' mapeou para '{col_value}', mas essa coluna não existe no df."
        )

# -------------------------
# 4) Mapear colunas (idade + 6 + MMSE_total + GENERO + SIT_TO_STAND)
# -------------------------
ID_COL = "ID_Aluno"
if ID_COL not in df.columns:
    raise ValueError("Coluna obrigatória 'ID_Aluno' não existe no df.")

DIST_COL       = pick_col(df, "6_MIN_ANDAR_Distancia_TOTAL_metros", ["6_MIN", "Distancia"])
TUG_COL        = pick_col(df, "Best_TIMED_UP_AND_GO_Simples", ["TIMED_UP_AND_GO", "Simples"])
HEIGHT_COL     = pick_col(df, "Antropometria_Estatura_cm", ["Estatura"])
MMSE_ESC_COL   = pick_col_any(df, ["MMSE_Escolaridade"], ["mmse", "escolar"])
MMSE_TOTAL_COL = pick_col_any(df, ["MMSE_total", "MMSE_Total", "MMSETOTAL"], ["mmse", "total", "score"])
AGE_COL        = pick_col_any(df, ["Idade", "Idade_anos", "IDADE", "Age", "AGE"], ["idade", "age"])

WEIGHT_COL     = pick_col_any(
    df,
    ["Antropometria_Peso_kg", "Antropometria_Peso", "Peso_kg", "PESO_kg", "Peso", "Weight_kg", "Weight"],
    ["peso", "weight", "massa"]
)

# NOVO: Sit-to-stand reps
STS_COL = pick_col_any(
    df,
    ["SIT_TO_STAND_Repetições", "SIT_TO_STAND_Repeticoes", "SIT_TO_STAND_Repeticoes_total", "SIT_TO_STAND_Repeticoes_Total",
     "SIT_TO_STAND_Repeticoes", "SIT_TO_STAND_Repetiçoes", "SIT_TO_STAND_Repeticoes", "SIT_TO_STAND_Repeticoes"],
    ["sit_to_stand", "repet", "stand"]
)

# NOVO: coluna de género/sexo (mantém valores originais)
GEN_COL = pick_col_any(
    df,
    ["Genero", "Género", "Sexo", "SEXO", "Sex", "sex", "Gender", "gender"],
    ["genero", "género", "sexo", "sex", "gender"]
)

# erros claros
assert_mapped("DIST_COL (6_MIN distância)", DIST_COL, df, ["6_min", "dist", "distancia"])
assert_mapped("TUG_COL (Timed Up and Go)", TUG_COL, df, ["timed_up_and_go", "tug"])
assert_mapped("HEIGHT_COL (Estatura)", HEIGHT_COL, df, ["estatura", "altura", "height"])
assert_mapped("MMSE_ESC_COL (MMSE escolaridade)", MMSE_ESC_COL, df, ["mmse", "escolar"])
assert_mapped("MMSE_TOTAL_COL (MMSE total)", MMSE_TOTAL_COL, df, ["mmse", "total", "score"])
assert_mapped("AGE_COL (Idade)", AGE_COL, df, ["idade", "age"])
assert_mapped("WEIGHT_COL (Peso)", WEIGHT_COL, df, ["peso", "weight", "massa"])
assert_mapped("GEN_COL (Genero/Sexo)", GEN_COL, df, ["genero", "género", "sexo", "sex", "gender"])
assert_mapped("STS_COL (Sit-to-Stand reps)", STS_COL, df, ["sit_to_stand", "repet", "stand"])

for var_name, col_value in [
    ("DIST_COL", DIST_COL),
    ("TUG_COL", TUG_COL),
    ("HEIGHT_COL", HEIGHT_COL),
    ("MMSE_ESC_COL", MMSE_ESC_COL),
    ("MMSE_TOTAL_COL", MMSE_TOTAL_COL),
    ("AGE_COL", AGE_COL),
    ("WEIGHT_COL", WEIGHT_COL),
    ("GEN_COL", GEN_COL),
    ("STS_COL", STS_COL),
]:
    assert_exists(var_name, col_value, df)

# -------------------------
# 5) Filtrar colunas + 1 linha por ID_Aluno
# -------------------------
num_cols = [
    AGE_COL, DIST_COL, TUG_COL, "HANDGRIP_BEST", HEIGHT_COL,
    MMSE_ESC_COL, MMSE_TOTAL_COL, WEIGHT_COL, STS_COL
]
keep_cols = [ID_COL, GEN_COL] + num_cols + (["Momento"] if "Momento" in df.columns else [])

tmp = df[keep_cols].copy()

for c in num_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

# scoring de completude (inclui GEN_COL sem mexer nos valores)
score_cols = [GEN_COL] + num_cols
tmp["_n_nonnull"] = tmp[score_cols].notna().sum(axis=1)

sort_cols = [ID_COL, "_n_nonnull"]
ascending = [True, False]
if "Momento" in tmp.columns:
    sort_cols.append("Momento")
    ascending.append(False)

tmp = (tmp.sort_values(sort_cols, ascending=ascending)
          .drop_duplicates(subset=[ID_COL], keep="first")
          .drop(columns=["_n_nonnull"])
          .reset_index(drop=True))

# manter o nome original da coluna de género como "Genero" no df_final
df_final = tmp[[ID_COL] + ["Genero"] + num_cols].copy()
df_final["Genero"] = tmp[GEN_COL]

# -------------------------
# 6) Ver dataset
# -------------------------
df_final.head(20)


,ID_Aluno,Genero,Age,@6_MIN_ANDAR_Distancia_TOTAL_metros,Best_TIMED_UP_AND_GO_Simples,HANDGRIP_BEST,ANTROPOMETRIA_Estatura_cm,Escolaridade,MMSE_Total,ANTROPOMETRIA_Peso_Kg,SIT_TO_STAND_Repetiçoes
0,19,0,81,495,8.450,12.8,152.0,10,20,57.7,15
1,37,0,74,360,10.420,15.0,150.0,4,17,78.2,11
2,42,1,84,247,12.840,19.0,157.0,4,23,85.0,9
3,80,0,78,295,12.260,11.0,145.0,4,26,76.6,9
4,81,1,69,455,5.560,30.0,161.0,9,29,77.8,15
5,136,0,73,554,6.000,17.0,148.0,4,27,54.0,14
6,182,0,70,520,6.650,10.0,149.0,6,26,57.0,14
7,206,0,70,450,6.720,15.0,151.0,4,27,58.6,12
8,279,0,68,494,7.430,18.0,148.0,9,30,68.2,14
9,284,0,60,350,8.500,20.0,156.0,6,25,87.7,9


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 0) Helpers: detetar colunas no df_final + erros úteis
# ============================================================
def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def candidate_cols(df, keywords_any, max_show=25):
    keys = [k.lower() for k in keywords_any]
    return [c for c in df.columns if any(k in c.lower() for k in keys)][:max_show]

def require_col(name, col, df, keywords):
    if col is None:
        raise ValueError(
            f"[TABLE ERROR] Não encontrei '{name}'.\n"
            f"  - Procurei por: {keywords}\n"
            f"  - Candidatos: {candidate_cols(df, keywords)}"
        )

# 1) Mapear colunas no df_final
ID_COL = "ID_Aluno" if "ID_Aluno" in df_final.columns else None

GEN_COL = "Genero" if "Genero" in df_final.columns else pick_col_any(df_final, ["Genero","Género","Sexo","Sex","gender"], ["genero","género","sexo","sex","gender"])
AGE_COL = pick_col_any(df_final, ["Idade","Idade_anos","Age"], ["idade","age"])
HEIGHT_COL = pick_col_any(df_final, ["Antropometria_Estatura_cm","Estatura_cm","Estatura","Height_cm","Height"], ["estatura","altura","height"])
HG_COL = "HANDGRIP_BEST" if "HANDGRIP_BEST" in df_final.columns else pick_col_any(df_final, ["Handgrip_best","Handgrip","HandgripStrength"], ["handgrip"])

require_col("Genero", GEN_COL, df_final, ["genero","género","sexo","sex","gender"])
require_col("Idade", AGE_COL, df_final, ["idade","age"])
require_col("Estatura (cm)", HEIGHT_COL, df_final, ["estatura","altura","height"])
require_col("HANDGRIP_BEST", HG_COL, df_final, ["handgrip"])

# 2) Preparar dados (numéricos + limpar)
df_t = df_final.copy()

df_t[AGE_COL] = pd.to_numeric(df_t[AGE_COL], errors="coerce")
df_t[HEIGHT_COL] = pd.to_numeric(df_t[HEIGHT_COL], errors="coerce")
df_t[HG_COL] = pd.to_numeric(df_t[HG_COL], errors="coerce")

# heurística: se estatura estiver em metros, converter para cm
med_h = df_t[HEIGHT_COL].median(skipna=True)
if pd.notna(med_h) and med_h < 3:
    df_t[HEIGHT_COL] = df_t[HEIGHT_COL] * 100

# manter só linhas completas para esta tabela
df_t = df_t.dropna(subset=[GEN_COL, AGE_COL, HEIGHT_COL, HG_COL]).copy()

# normalizar genero para "Women/Men" mantendo 0/1 como origem (0=Women, 1=Men)
# (não muda o df_final; só para labels da tabela)
df_t["Sex_group"] = df_t[GEN_COL].astype(str).str.strip()
df_t["Sex_group"] = df_t["Sex_group"].replace({"0": "Women", "1": "Men"})

# total por sexo (para percentagens)
sex_totals = df_t["Sex_group"].value_counts().to_dict()

# 3) Definir grupos de idade e estatura (como no artigo)
age_bins = [65, 75, 85, np.inf]
age_labels = ["[65–75[", "[75–85[", "≥85"]
df_t["Age_group"] = pd.cut(df_t[AGE_COL], bins=age_bins, right=False, labels=age_labels)

def height_group(row):
    h = row[HEIGHT_COL]
    s = row["Sex_group"]
    if pd.isna(h) or pd.isna(s):
        return np.nan

    if s == "Women":
        # <148, [148–153[, ≥153
        if h < 148: return "<148"
        if 148 <= h < 153: return "[148–153["
        return "≥153"
    else:
        # Men: <161, [161–167[, ≥167
        if h < 161: return "<161"
        if 161 <= h < 167: return "[161–167["
        return "≥167"

df_t["Height_group"] = df_t.apply(height_group, axis=1)

# manter só linhas que caem nos bins de idade (>=65) e que tenham grupos definidos
df_t = df_t.dropna(subset=["Age_group", "Height_group"]).copy()

# 4) Função para stats por grupo (formato tabela)
def group_stats(x: pd.Series):
    x = x.dropna()
    if len(x) == 0:
        return {
            "mean_sd": "",
            "p85mean": "",
            "minmax": "",
            "P10": "", "P15": "", "P25": "", "P50": "", "P75": "", "P85": "", "P90": ""
        }
    mean = x.mean()
    sd = x.std(ddof=1)
    p85mean = 0.85 * mean
    mn, mx = x.min(), x.max()
    qs = x.quantile([0.10,0.15,0.25,0.50,0.75,0.85,0.90])

    return {
        "mean_sd": f"{mean:.1f} ({sd:.1f})" if pd.notna(sd) else f"{mean:.1f} (NA)",
        "p85mean": f"{p85mean:.1f}",
        "minmax": f"{mn:.1f}–{mx:.1f}",
        "P10": f"{qs.loc[0.10]:.1f}",
        "P15": f"{qs.loc[0.15]:.1f}",
        "P25": f"{qs.loc[0.25]:.1f}",
        "P50": f"{qs.loc[0.50]:.1f}",
        "P75": f"{qs.loc[0.75]:.1f}",
        "P85": f"{qs.loc[0.85]:.1f}",
        "P90": f"{qs.loc[0.90]:.1f}",
    }

# 5) Construir tabela final
rows = []
sex_order = ["Women", "Men"]
age_order = age_labels

# ordem de altura depende do sexo
height_order_w = ["<148", "[148–153[", "≥153"]
height_order_m = ["<161", "[161–167[", "≥167"]

for sex in sex_order:
    df_sex = df_t[df_t["Sex_group"] == sex].copy()
    n_sex = len(df_sex)

    # linha de cabeçalho tipo "Women, n = ..."
    rows.append({
        "Sex": f"{sex}, n = {n_sex}",
        "Age range (years)": "",
        "Height range (cm)": "",
        "n (%)": "",
        "mean (SD)": "",
        "85% of mean": "",
        "min–max": "",
        "P10": "", "P15": "", "P25": "", "P50": "", "P75": "", "P85": "", "P90": ""
    })

    for ageg in age_order:
        df_age = df_sex[df_sex["Age_group"] == ageg].copy()

        h_order = height_order_w if sex == "Women" else height_order_m

        first_in_age = True
        for hg in h_order:
            df_cell = df_age[df_age["Height_group"] == hg]
            n = len(df_cell)
            pct = (n / n_sex * 100) if n_sex > 0 else np.nan

            stats = group_stats(df_cell[HG_COL])

            rows.append({
                "Sex": "",
                "Age range (years)": ageg if first_in_age else "",
                "Height range (cm)": hg,
                "n (%)": f"{n} ({pct:.1f})" if n_sex > 0 else f"{n} (NA)",
                "mean (SD)": stats["mean_sd"],
                "85% of mean": stats["p85mean"],
                "min–max": stats["minmax"],
                "P10": stats["P10"],
                "P15": stats["P15"],
                "P25": stats["P25"],
                "P50": stats["P50"],
                "P75": stats["P75"],
                "P85": stats["P85"],
                "P90": stats["P90"],
            })

            first_in_age = False

table_handgrip = pd.DataFrame(rows, columns=[
    "Sex",
    "Age range (years)",
    "Height range (cm)",
    "n (%)",
    "mean (SD)",
    "85% of mean",
    "min–max",
    "P10","P15","P25","P50","P75","P85","P90"
])

table_handgrip




,Sex,Age range (years),Height range (cm),n (%),mean (SD),85% of mean,min–max,P10,P15,P25,P50,P75,P85,P90
0,"Women, n = 1638",,,,,,,,,,,,,
1,,[65–75[,<148,232 (14.2),18.6 (5.8),15.8,2.0–40.0,11.0,14.0,16.0,19.2,22.0,23.5,25.0
2,,,[148–153[,365 (22.3),19.9 (5.4),17.0,1.0–36.0,14.0,15.4,17.4,20.3,23.5,25.0,26.2
3,,,≥153,581 (35.5),21.3 (6.4),18.1,1.0–60.0,15.0,16.0,18.7,22.0,24.6,26.0,27.2
4,,[75–85[,<148,109 (6.7),16.6 (5.5),14.1,2.0–29.6,10.0,12.0,14.0,17.7,20.0,21.2,22.1
5,,,[148–153[,148 (9.0),17.8 (5.4),15.1,1.0–28.0,11.4,14.0,15.4,18.0,21.5,23.0,23.9
6,,,≥153,175 (10.7),20.5 (5.9),17.4,1.0–60.0,15.0,16.0,17.0,20.0,24.0,25.7,27.0
7,,≥85,<148,12 (0.7),16.4 (3.7),14.0,8.5–21.9,12.3,13.9,15.8,16.3,17.8,20.4,20.9
8,,,[148–153[,8 (0.5),12.9 (5.8),11.0,4.0–19.8,5.4,6.2,9.8,13.4,17.1,19.4,19.7
9,,,≥153,8 (0.5),18.5 (3.4),15.7,14.0–22.9,14.0,14.1,15.5,20.0,20.4,21.0,21.6


In [19]:
import pandas as pd

ref_rows = [
    # ---------------------------
    # Women
    # ---------------------------
    # Age [65–75[
    dict(Sex_group="Women", Age_group="[65–75[", Height_group="<148",
         ref_mean=18.7, ref_sd=4.6, P10=12.6, P15=14.1, P25=16.3, P50=18.1, P75=21.9, P85=22.9, P90=25.1),
    dict(Sex_group="Women", Age_group="[65–75[", Height_group="[148–153[",
         ref_mean=19.8, ref_sd=5.5, P10=12.5, P15=14.3, P25=16.9, P50=20.5, P75=23.1, P85=24.6, P90=25.9),
    dict(Sex_group="Women", Age_group="[65–75[", Height_group="≥153",
         ref_mean=21.1, ref_sd=5.5, P10=14.3, P15=15.2, P25=17.0, P50=21.0, P75=25.4, P85=27.0, P90=28.3),

    # Age [75–85[
    dict(Sex_group="Women", Age_group="[75–85[", Height_group="<148",
         ref_mean=15.3, ref_sd=4.1, P10=10.2, P15=10.9, P25=12.7, P50=15.1, P75=17.9, P85=19.8, P90=20.7),
    dict(Sex_group="Women", Age_group="[75–85[", Height_group="[148–153[",
         ref_mean=16.8, ref_sd=4.7, P10=9.9, P15=12.1, P25=14.3, P50=16.5, P75=19.9, P85=22.1, P90=22.9),
    dict(Sex_group="Women", Age_group="[75–85[", Height_group="≥153",
         ref_mean=17.9, ref_sd=4.7, P10=11.8, P15=12.8, P25=15.5, P50=17.6, P75=21.6, P85=23.0, P90=23.7),

    # Age ≥85
    dict(Sex_group="Women", Age_group="≥85", Height_group="<148",
         ref_mean=13.4, ref_sd=3.8, P10=8.6, P15=9.4, P25=10.5, P50=13.3, P75=15.9, P85=17.5, P90=18.3),
    dict(Sex_group="Women", Age_group="≥85", Height_group="[148–153[",
         ref_mean=14.8, ref_sd=3.7, P10=9.6, P15=10.2, P25=11.1, P50=15.1, P75=17.7, P85=19.1, P90=19.5),
    dict(Sex_group="Women", Age_group="≥85", Height_group="≥153",
         ref_mean=16.9, ref_sd=3.9, P10=11.3, P15=12.2, P25=14.4, P50=18.0, P75=19.4, P85=22.1, P90=22.7),

    # ---------------------------
    # Men
    # ---------------------------
    # Age [65–75[
    dict(Sex_group="Men", Age_group="[65–75[", Height_group="<161",
         ref_mean=28.6, ref_sd=7.9, P10=16.9, P15=18.8, P25=23.7, P50=29.3, P75=34.5, P85=35.1, P90=38.2),
    dict(Sex_group="Men", Age_group="[65–75[", Height_group="[161–167[",
         ref_mean=32.6, ref_sd=8.4, P10=20.5, P15=23.8, P25=26.3, P50=32.8, P75=38.9, P85=41.8, P90=43.8),
    dict(Sex_group="Men", Age_group="[65–75[", Height_group="≥167",
         ref_mean=36.9, ref_sd=9.2, P10=23.9, P15=27.3, P25=31.1, P50=38.5, P75=43.9, P85=45.8, P90=47.3),

    # Age [75–85[
    dict(Sex_group="Men", Age_group="[75–85[", Height_group="<161",
         ref_mean=25.5, ref_sd=7.7, P10=16.3, P15=17.4, P25=20.8, P50=25.9, P75=30.1, P85=33.6, P90=34.9),
    dict(Sex_group="Men", Age_group="[75–85[", Height_group="[161–167[",
         ref_mean=27.5, ref_sd=6.8, P10=19.4, P15=20.4, P25=23.6, P50=27.4, P75=32.1, P85=34.0, P90=35.2),
    dict(Sex_group="Men", Age_group="[75–85[", Height_group="≥167",
         ref_mean=30.4, ref_sd=6.4, P10=23.0, P15=24.7, P25=25.6, P50=30.9, P75=34.2, P85=38.8, P90=40.2),

    # Age ≥85
    dict(Sex_group="Men", Age_group="≥85", Height_group="<161",
         ref_mean=19.1, ref_sd=4.6, P10=13.5, P15=14.5, P25=17.4, P50=19.1, P75=21.5, P85=22.6, P90=25.2),
    dict(Sex_group="Men", Age_group="≥85", Height_group="[161–167[",
         ref_mean=23.9, ref_sd=6.2, P10=14.7, P15=16.3, P25=19.8, P50=24.5, P75=27.4, P85=30.3, P90=34.5),
    dict(Sex_group="Men", Age_group="≥85", Height_group="≥167",
         ref_mean=29.2, ref_sd=9.0, P10=21.2, P15=21.3, P25=21.3, P50=26.1, P75=32.8, P85=45.8, P90=45.9),
]

ref_table = pd.DataFrame(ref_rows)

# sanity check
assert ref_table.shape[0] == 18
ref_table.head()



,Sex_group,Age_group,Height_group,ref_mean,ref_sd,P10,P15,P25,P50,P75,P85,P90
0,Women,[65–75[,<148,18.7,4.6,12.6,14.1,16.3,18.1,21.9,22.9,25.1
1,Women,[65–75[,[148–153[,19.8,5.5,12.5,14.3,16.9,20.5,23.1,24.6,25.9
2,Women,[65–75[,≥153,21.1,5.5,14.3,15.2,17.0,21.0,25.4,27.0,28.3
3,Women,[75–85[,<148,15.3,4.1,10.2,10.9,12.7,15.1,17.9,19.8,20.7
4,Women,[75–85[,[148–153[,16.8,4.7,9.9,12.1,14.3,16.5,19.9,22.1,22.9


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon, binomtest

# ---- helpers to detect columns in df_final ----
def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def candidate_cols(df, keywords_any, max_show=25):
    keys = [k.lower() for k in keywords_any]
    return [c for c in df.columns if any(k in c.lower() for k in keys)][:max_show]

def require_col(name, col, df, keywords):
    if col is None:
        raise ValueError(
            f"[COMPARE ERROR] Could not find '{name}'.\n"
            f"  - searched for: {keywords}\n"
            f"  - candidates: {candidate_cols(df, keywords)}"
        )

# ---- map columns in df_final ----
GEN_COL = "Genero" if "Genero" in df_final.columns else pick_col_any(df_final, ["Genero","Género","Sexo","Sex","gender"], ["genero","género","sexo","sex","gender"])
AGE_COL = pick_col_any(df_final, ["Idade","Idade_anos","Age"], ["idade","age"])
HEIGHT_COL = pick_col_any(df_final, ["Antropometria_Estatura_cm","Estatura_cm","Estatura","Height_cm","Height"], ["estatura","altura","height"])
HG_COL = "HANDGRIP_BEST" if "HANDGRIP_BEST" in df_final.columns else pick_col_any(df_final, ["Handgrip_best","Handgrip"], ["handgrip"])

require_col("Genero", GEN_COL, df_final, ["genero","género","sexo","sex","gender"])
require_col("Idade", AGE_COL, df_final, ["idade","age"])
require_col("Estatura", HEIGHT_COL, df_final, ["estatura","altura","height"])
require_col("HANDGRIP_BEST", HG_COL, df_final, ["handgrip"])

# ---- prep your data ----
df = df_final.copy()
df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")
df[HEIGHT_COL] = pd.to_numeric(df[HEIGHT_COL], errors="coerce")
df[HG_COL] = pd.to_numeric(df[HG_COL], errors="coerce")

# heuristic: if height in meters, convert to cm
med_h = df[HEIGHT_COL].median(skipna=True)
if pd.notna(med_h) and med_h < 3:
    df[HEIGHT_COL] = df[HEIGHT_COL] * 100

df = df.dropna(subset=[GEN_COL, AGE_COL, HEIGHT_COL, HG_COL]).copy()

# only keep subjects in the paper's age-range definition (>=65)
df = df[df[AGE_COL] >= 65].copy()

# map 0/1 -> Women/Men (only for matching reference strata)
df["Sex_group"] = df[GEN_COL].astype(str).str.strip().replace({"0": "Women", "1": "Men"})

# age groups
age_bins = [65, 75, 85, np.inf]
age_labels = ["[65–75[", "[75–85[", "≥85"]
df["Age_group"] = pd.cut(df[AGE_COL], bins=age_bins, right=False, labels=age_labels)

# height group (sex-specific)
def height_group(row):
    h = row[HEIGHT_COL]
    s = row["Sex_group"]
    if pd.isna(h) or pd.isna(s):
        return np.nan
    if s == "Women":
        if h < 148: return "<148"
        if 148 <= h < 153: return "[148–153["
        return "≥153"
    else:  # Men
        if h < 161: return "<161"
        if 161 <= h < 167: return "[161–167["
        return "≥167"

df["Height_group"] = df.apply(height_group, axis=1)
df = df.dropna(subset=["Sex_group","Age_group","Height_group"]).copy()

# ---- merge with reference ----
need_cols = ["Sex_group","Age_group","Height_group","ref_mean","ref_sd","P10","P15","P25","P50","P75","P85","P90"]
missing_ref = [c for c in need_cols if c not in ref_table.columns]
if missing_ref:
    raise ValueError(f"[COMPARE ERROR] ref_table is missing columns: {missing_ref}")

m = df.merge(ref_table[need_cols], on=["Sex_group","Age_group","Height_group"], how="left")

if m["ref_mean"].isna().any():
    bad = m[m["ref_mean"].isna()][["Sex_group","Age_group","Height_group"]].drop_duplicates()
    raise ValueError(f"[COMPARE ERROR] Some strata not found in ref_table:\n{bad}")

# ============================================================
# Tests
# ============================================================
x = m[HG_COL].astype(float)
n = len(m)

# (1) z-score test (mean(z)=0 under reference)
m["z"] = (x - m["ref_mean"]) / m["ref_sd"]
z = m["z"].dropna()
t_z = ttest_1samp(z, popmean=0.0)

# (2) binomial tests: proportion below P10 / P25
below_p10 = int((x < m["P10"]).sum())
below_p25 = int((x < m["P25"]).sum())
binom_p10 = binomtest(below_p10, n=n, p=0.10, alternative="two-sided")
binom_p25 = binomtest(below_p25, n=n, p=0.25, alternative="two-sided")

# (3) percentile-rank (approx via interpolation between P10..P90)
pcts = np.array([10, 15, 25, 50, 75, 85, 90], dtype=float)

def interp_percentile(row):
    vals = np.array([row["P10"], row["P15"], row["P25"], row["P50"], row["P75"], row["P85"], row["P90"]], dtype=float)
    hg = float(row[HG_COL])

    # handle weird/missing
    if np.any(~np.isfinite(vals)) or not np.isfinite(hg):
        return np.nan

    # clamp to [10, 90]
    if hg <= vals[0]: return 10.0
    if hg >= vals[-1]: return 90.0
    return float(np.interp(hg, vals, pcts))

m["pct_rank"] = m.apply(interp_percentile, axis=1)
pct = m["pct_rank"].dropna()

# test median percentile = 50
w_pct = wilcoxon(pct - 50.0, zero_method="wilcox", correction=False)

# Summary output + quick interpretation
print(f"N matched to reference strata: {n}")

print("\n(1) Z-score one-sample t-test vs 0")
print(f"  mean(z) = {z.mean():.3f}  (positive => stronger; negative => weaker)")
print(f"  t = {t_z.statistic:.3f}, p = {t_z.pvalue:.4g}")

print("\n(2) Binomial tests vs expected % below reference cutoffs")
print(f"  Below P10: {below_p10}/{n} = {below_p10/n:.3%} | p = {binom_p10.pvalue:.4g}")
print(f"  Below P25: {below_p25}/{n} = {below_p25/n:.3%} | p = {binom_p25.pvalue:.4g}")

print("\n(3) Percentile-rank test (Wilcoxon) vs median=50")
print(f"  median(pct_rank) = {np.median(pct):.1f}  ( >50 stronger ; <50 weaker )")
print(f"  W = {w_pct.statistic:.3f}, p = {w_pct.pvalue:.4g}")

# Optional: per-sex breakdown (same tests, quick)
for sex in ["Women", "Men"]:
    ms = m[m["Sex_group"] == sex].copy()
    if len(ms) < 5:
        continue
    xs = ms[HG_COL].astype(float)
    zs = ((xs - ms["ref_mean"]) / ms["ref_sd"]).dropna()
    tzs = ttest_1samp(zs, 0.0)
    bp10 = int((xs < ms["P10"]).sum())
    bp25 = int((xs < ms["P25"]).sum())
    nsex = len(ms)
    print(f"\n--- {sex} (n={nsex}) ---")
    print(f"mean(z)={zs.mean():.3f} | p={tzs.pvalue:.4g}")
    print(f"Below P10: {bp10/nsex:.1%} | Below P25: {bp25/nsex:.1%}")


N matched to reference strata: 2457

(1) Z-score one-sample t-test vs 0
  mean(z) = 0.069  (positive => stronger; negative => weaker)
  t = 2.920, p = 0.003535

(2) Binomial tests vs expected % below reference cutoffs
  Below P10: 225/2457 = 9.158% | p = 0.1681
  Below P25: 491/2457 = 19.984% | p = 4.759e-09

(3) Percentile-rank test (Wilcoxon) vs median=50
  median(pct_rank) = 55.7  ( >50 stronger ; <50 weaker )
  W = 1253778.000, p = 4.692e-11

--- Women (n=1638) ---
mean(z)=0.119 | p=4.609e-05
Below P10: 8.7% | Below P25: 19.4%

--- Men (n=819) ---
mean(z)=-0.031 | p=0.4432
Below P10: 10.1% | Below P25: 21.1%


“Using Portuguese normative percentiles stratified by sex, age and height, our cohort showed slightly higher handgrip strength overall (median percentile 55.7; p<1e−10), driven mainly by women (mean z=0.119; p<1e−4). The proportion below the 10th percentile was not different from expected, while the proportion below the 25th percentile was significantly lower, suggesting fewer individuals with low handgrip strength.”

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway, kruskal

# ----------------------------
# 0) Helpers: detetar colunas
# ----------------------------
def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def candidate_cols(df, keywords_any, max_show=25):
    keys = [k.lower() for k in keywords_any]
    return [c for c in df.columns if any(k in c.lower() for k in keys)][:max_show]

def require_col(name, col, df, keywords):
    if col is None:
        raise ValueError(
            f"[TABLE1 ERROR] Não encontrei '{name}'.\n"
            f"  - Procurei por: {keywords}\n"
            f"  - Candidatos: {candidate_cols(df, keywords)}"
        )

# 1) Mapear colunas no df_final (SEM MMSE)
GEN_COL = "Genero" if "Genero" in df_final.columns else pick_col_any(
    df_final, ["Genero","Género","Sexo","Sex","gender"], ["genero","género","sexo","sex","gender"]
)
AGE_COL = pick_col_any(df_final, ["Idade","Idade_anos","Age"], ["idade","age"])
W_COL   = pick_col_any(df_final, ["Antropometria_Peso_kg","Peso_kg","Peso","Weight_kg","Weight"], ["peso","weight","massa"])
H_COL   = pick_col_any(df_final, ["Antropometria_Estatura_cm","Estatura_cm","Estatura","Height_cm","Height"], ["estatura","altura","height"])
HG_COL  = "HANDGRIP_BEST" if "HANDGRIP_BEST" in df_final.columns else pick_col_any(df_final, ["Handgrip_best","Handgrip"], ["handgrip"])

for name, col, keys in [
    ("Genero", GEN_COL, ["genero","género","sexo","sex","gender"]),
    ("Idade", AGE_COL, ["idade","age"]),
    ("Peso", W_COL, ["peso","weight","massa"]),
    ("Estatura", H_COL, ["estatura","altura","height"]),
    ("HANDGRIP_BEST", HG_COL, ["handgrip"]),
]:
    require_col(name, col, df_final, keys)

# 2) Preparar dataset
# ---------------------------
df = df_final.copy()

df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")
df[W_COL]   = pd.to_numeric(df[W_COL], errors="coerce")
df[H_COL]   = pd.to_numeric(df[H_COL], errors="coerce")
df[HG_COL]  = pd.to_numeric(df[HG_COL], errors="coerce")

# estatura: se estiver em metros, converter para cm
med_h = df[H_COL].median(skipna=True)
if pd.notna(med_h) and med_h < 3:
    df[H_COL] = df[H_COL] * 100

# BMI
df["BMI"] = df[W_COL] / ((df[H_COL] / 100) ** 2)

# Sexo (0/1) -> Women/Men só para labels da tabela
df["Sex_group"] = df[GEN_COL].astype(str).str.strip().replace({"0": "Women", "1": "Men"})

# Grupos etários (como no paper)
age_bins = [65, 75, 85, np.inf]
age_labels = ["[65–75[", "[75–85[", "≥85"]
df["Age_group"] = pd.cut(df[AGE_COL], bins=age_bins, right=False, labels=age_labels)

# Manter apenas linhas com info mínima para aparecer na tabela
df = df.dropna(subset=["Sex_group", "Age_group"]).copy()

# 3) Funções de resumo + p-values
def mean_sd(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return ""
    return f"{s.mean():.1f} ({s.std(ddof=1):.1f})"

def n_pct(n, total):
    if total == 0:
        return "0 (NA)"
    return f"{n} ({(n/total)*100:.1f})"

def p_continuous(data_by_group, method="anova"):
    groups = [g.dropna().astype(float).values for g in data_by_group if g.dropna().shape[0] > 1]
    if len(groups) < 2:
        return ""
    if method == "kruskal":
        return f"{kruskal(*groups).pvalue:.3g}"
    return f"{f_oneway(*groups).pvalue:.3g}"

CONT_METHOD = "anova"   # ou "kruskal"

# 4) Construir tabela estilo paper (SEM educação/MMSE)
sex_order = ["Women", "Men"]
age_order = age_labels
table_rows = []

for sex in sex_order:
    dsex = df[df["Sex_group"] == sex].copy()
    total_sex = len(dsex)

    # linha n (%)
    counts = dsex["Age_group"].value_counts().reindex(age_order).fillna(0).astype(int)
    row = {"Variable": "n (%)", "p": ""}
    for ag in age_order:
        row[ag] = n_pct(int(counts.loc[ag]), total_sex)
    table_rows.append((sex, row))

    # variáveis contínuas
    cont_vars = [
        ("Weight (kg), mean (SD)", W_COL),
        ("Height (cm), mean (SD)", H_COL),
        ("Body mass index (kg/m²), mean (SD)", "BMI"),
        ("Handgrip strength (kgf), mean (SD)", HG_COL),
    ]

    for label, col in cont_vars:
        row = {"Variable": label}
        data_by_group = [dsex.loc[dsex["Age_group"] == ag, col] for ag in age_order]
        row["p"] = p_continuous(data_by_group, method=CONT_METHOD)
        for ag in age_order:
            row[ag] = mean_sd(dsex.loc[dsex["Age_group"] == ag, col])
        table_rows.append((sex, row))

# lado a lado (Women vs Men)
women_rows = [r for s, r in table_rows if s == "Women"]
men_rows   = [r for s, r in table_rows if s == "Men"]

tab_w = pd.DataFrame(women_rows)[["Variable"] + age_order + ["p"]].rename(columns={"p": "p (Women)"})
tab_m = pd.DataFrame(men_rows)[age_order + ["p"]].rename(columns={ag: f"{ag} (Men)" for ag in age_order}).rename(columns={"p": "p (Men)"})

table1_like = pd.concat([tab_w, tab_m], axis=1)

table1_like


,Variable,[65–75[,[75–85[,≥85,p (Women),[65–75[ (Men),[75–85[ (Men),≥85 (Men),p (Men)
0,n (%),1178 (71.9),432 (26.4),28 (1.7),,525 (64.1),269 (32.8),25 (3.1),
1,"Weight (kg), mean (SD)",69.6 (10.8),68.0 (9.7),61.9 (10.2),5.28e-05,78.2 (11.0),76.0 (9.7),73.9 (10.0),0.00441
2,"Height (cm), mean (SD)",152.4 (5.6),151.3 (5.2),149.7 (6.3),9.16e-05,165.1 (5.9),163.4 (5.6),162.4 (5.9),0.000125
3,"Body mass index (kg/m²), mean (SD)",30.0 (4.5),29.7 (4.0),27.5 (3.6),0.0117,28.7 (3.6),28.5 (3.4),28.0 (3.7),0.503
4,"Handgrip strength (kgf), mean (SD)",20.3 (6.1),18.6 (5.9),16.0 (4.7),7.58e-09,31.8 (9.9),29.3 (7.9),25.9 (8.7),4.82e-05


In [ ]:
import pandas as pd
import numpy as np

# Reference (Table 1 do paper) — SÓ 4 LINHAS
# (Weight, Height, BMI, Handgrip) por Sexo x Age_group

age_groups = ["[65–75[", "[75–85[", "≥85"]
sexes = ["Women", "Men"]

cols = pd.MultiIndex.from_product(
    [sexes, age_groups, ["mean", "sd"]],
    names=["Sex_group", "Age_group", "stat"]
)

ref_table1_wide = pd.DataFrame(index=[
    "Weight (kg)",
    "Height (cm)",
    "BMI (kg/m²)",
    "Handgrip (kgf)"
], columns=cols, dtype=float)

# ---- Women ----
ref_table1_wide.loc["Weight (kg)",   ("Women","[65–75[","mean")] = 70.3
ref_table1_wide.loc["Weight (kg)",   ("Women","[65–75[","sd")]   = 12.8
ref_table1_wide.loc["Weight (kg)",   ("Women","[75–85[","mean")] = 68.6
ref_table1_wide.loc["Weight (kg)",   ("Women","[75–85[","sd")]   = 12.6
ref_table1_wide.loc["Weight (kg)",   ("Women","≥85","mean")]     = 62.3
ref_table1_wide.loc["Weight (kg)",   ("Women","≥85","sd")]       = 11.2

ref_table1_wide.loc["Height (cm)",   ("Women","[65–75[","mean")] = 152.9
ref_table1_wide.loc["Height (cm)",   ("Women","[65–75[","sd")]   = 5.9
ref_table1_wide.loc["Height (cm)",   ("Women","[75–85[","mean")] = 150.5
ref_table1_wide.loc["Height (cm)",   ("Women","[75–85[","sd")]   = 5.9
ref_table1_wide.loc["Height (cm)",   ("Women","≥85","mean")]     = 147.3
ref_table1_wide.loc["Height (cm)",   ("Women","≥85","sd")]       = 5.7

ref_table1_wide.loc["BMI (kg/m²)",   ("Women","[65–75[","mean")] = 30.0
ref_table1_wide.loc["BMI (kg/m²)",   ("Women","[65–75[","sd")]   = 5.0
ref_table1_wide.loc["BMI (kg/m²)",   ("Women","[75–85[","mean")] = 30.2
ref_table1_wide.loc["BMI (kg/m²)",   ("Women","[75–85[","sd")]   = 5.1
ref_table1_wide.loc["BMI (kg/m²)",   ("Women","≥85","mean")]     = 28.6
ref_table1_wide.loc["BMI (kg/m²)",   ("Women","≥85","sd")]       = 4.6

ref_table1_wide.loc["Handgrip (kgf)",("Women","[65–75[","mean")] = 20.1
ref_table1_wide.loc["Handgrip (kgf)",("Women","[65–75[","sd")]   = 5.4
ref_table1_wide.loc["Handgrip (kgf)",("Women","[75–85[","mean")] = 16.6
ref_table1_wide.loc["Handgrip (kgf)",("Women","[75–85[","sd")]   = 4.6
ref_table1_wide.loc["Handgrip (kgf)",("Women","≥85","mean")]     = 14.3
ref_table1_wide.loc["Handgrip (kgf)",("Women","≥85","sd")]       = 3.9

# ---- Men ----
ref_table1_wide.loc["Weight (kg)",   ("Men","[65–75[","mean")] = 78.1
ref_table1_wide.loc["Weight (kg)",   ("Men","[65–75[","sd")]   = 12.2
ref_table1_wide.loc["Weight (kg)",   ("Men","[75–85[","mean")] = 77.5
ref_table1_wide.loc["Weight (kg)",   ("Men","[75–85[","sd")]   = 11.8
ref_table1_wide.loc["Weight (kg)",   ("Men","≥85","mean")]     = 72.6
ref_table1_wide.loc["Weight (kg)",   ("Men","≥85","sd")]       = 9.8

ref_table1_wide.loc["Height (cm)",   ("Men","[65–75[","mean")] = 165.9
ref_table1_wide.loc["Height (cm)",   ("Men","[65–75[","sd")]   = 6.8
ref_table1_wide.loc["Height (cm)",   ("Men","[75–85[","mean")] = 163.9
ref_table1_wide.loc["Height (cm)",   ("Men","[75–85[","sd")]   = 6.6
ref_table1_wide.loc["Height (cm)",   ("Men","≥85","mean")]     = 161.5
ref_table1_wide.loc["Height (cm)",   ("Men","≥85","sd")]       = 6.6

ref_table1_wide.loc["BMI (kg/m²)",   ("Men","[65–75[","mean")] = 28.3
ref_table1_wide.loc["BMI (kg/m²)",   ("Men","[65–75[","sd")]   = 3.9
ref_table1_wide.loc["BMI (kg/m²)",   ("Men","[75–85[","mean")] = 28.8
ref_table1_wide.loc["BMI (kg/m²)",   ("Men","[75–85[","sd")]   = 4.2
ref_table1_wide.loc["BMI (kg/m²)",   ("Men","≥85","mean")]     = 27.9
ref_table1_wide.loc["BMI (kg/m²)",   ("Men","≥85","sd")]       = 3.9

ref_table1_wide.loc["Handgrip (kgf)",("Men","[65–75[","mean")] = 33.4
ref_table1_wide.loc["Handgrip (kgf)",("Men","[65–75[","sd")]   = 9.3
ref_table1_wide.loc["Handgrip (kgf)",("Men","[75–85[","mean")] = 27.4
ref_table1_wide.loc["Handgrip (kgf)",("Men","[75–85[","sd")]   = 7.3
ref_table1_wide.loc["Handgrip (kgf)",("Men","≥85","mean")]     = 22.5
ref_table1_wide.loc["Handgrip (kgf)",("Men","≥85","sd")]       = 7.2

# Isto é a "tabela com 4 linhas" tipo paper
ref_table1_wide



Sex_group        Women                                      Men                \
Age_group      [65–75[       [75–85[          ≥85       [65–75[       [75–85[   
stat              mean    sd    mean    sd   mean    sd    mean    sd    mean   
Weight (kg)       70.3  12.8    68.6  12.6   62.3  11.2    78.1  12.2    77.5   
Height (cm)      152.9   5.9   150.5   5.9  147.3   5.7   165.9   6.8   163.9   
BMI (kg/m²)       30.0   5.0    30.2   5.1   28.6   4.6    28.3   3.9    28.8   
Handgrip (kgf)    20.1   5.4    16.6   4.6   14.3   3.9    33.4   9.3    27.4   

Sex_group                         
Age_group               ≥85       
stat              sd   mean   sd  
Weight (kg)     11.8   72.6  9.8  
Height (cm)      6.6  161.5  6.6  
BMI (kg/m²)      4.2   27.9  3.9  
Handgrip (kgf)   7.3   22.5  7.2

In [25]:
# Converte a tabela de 4 linhas (wide) para uma tabela por estrato (Sex x Age)
long = (ref_table1_wide
        .stack(["Sex_group","Age_group"])
        .reset_index()
        .rename(columns={"level_0":"Variable"}))

# long tem colunas: Variable, Sex_group, Age_group, mean, sd
pivot = long.pivot_table(index=["Sex_group","Age_group"], columns="Variable", values=["mean","sd"])

var_map = {
    "Weight (kg)": "w",
    "Height (cm)": "h",
    "BMI (kg/m²)": "bmi",
    "Handgrip (kgf)": "hg"
}

# flatten columns -> w_mean, w_sd, ...
pivot.columns = [f"{var_map[var]}_{stat}" for stat, var in pivot.columns]
ref_table1 = pivot.reset_index()

ref_table1


C:\Users\tiago1951\AppData\Local\Temp\ipykernel_23528\1834268916.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(["Sex_group","Age_group"])


,Sex_group,Age_group,bmi_mean,hg_mean,h_mean,w_mean,bmi_sd,hg_sd,h_sd,w_sd
0,Men,[65–75[,28.3,33.4,165.9,78.1,3.9,9.3,6.8,12.2
1,Men,[75–85[,28.8,27.4,163.9,77.5,4.2,7.3,6.6,11.8
2,Men,≥85,27.9,22.5,161.5,72.6,3.9,7.2,6.6,9.8
3,Women,[65–75[,30.0,20.1,152.9,70.3,5.0,5.4,5.9,12.8
4,Women,[75–85[,30.2,16.6,150.5,68.6,5.1,4.6,5.9,12.6
5,Women,≥85,28.6,14.3,147.3,62.3,4.6,3.9,5.7,11.2


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon

# 0) Detect columns in df_final
def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def require_col(name, col, df):
    if col is None or col not in df.columns:
        raise ValueError(f"[COMPARE ERROR] Missing column for '{name}'. Got: {col}")

GEN_COL = "Genero" if "Genero" in df_final.columns else pick_col_any(df_final, ["Genero","Género","Sexo","sex","Sex"], ["genero","género","sexo","sex"])
AGE_COL = pick_col_any(df_final, ["Idade","Idade_anos","Age"], ["idade","age"])
W_COL   = pick_col_any(df_final, ["Antropometria_Peso_kg","Peso_kg","Peso","Weight_kg","Weight"], ["peso","weight","massa"])
H_COL   = pick_col_any(df_final, ["Antropometria_Estatura_cm","Estatura_cm","Estatura","Height_cm","Height"], ["estatura","altura","height"])
HG_COL  = "HANDGRIP_BEST" if "HANDGRIP_BEST" in df_final.columns else pick_col_any(df_final, ["Handgrip_best","Handgrip"], ["handgrip"])

for nm, col in [("Genero",GEN_COL),("Idade",AGE_COL),("Peso",W_COL),("Estatura",H_COL),("Handgrip",HG_COL)]:
    require_col(nm, col, df_final)

# 1) Prepare cohort + strata (Sex x Age_group) and merge reference
d = df_final.copy()
d[AGE_COL] = pd.to_numeric(d[AGE_COL], errors="coerce")
d[W_COL]   = pd.to_numeric(d[W_COL], errors="coerce")
d[H_COL]   = pd.to_numeric(d[H_COL], errors="coerce")
d[HG_COL]  = pd.to_numeric(d[HG_COL], errors="coerce")

# height meters -> cm heuristic
med_h = d[H_COL].median(skipna=True)
if pd.notna(med_h) and med_h < 3:
    d[H_COL] = d[H_COL] * 100

# BMI
d["BMI"] = d[W_COL] / ((d[H_COL] / 100) ** 2)

# Sex_group for matching strata (Genero: 0 Women, 1 Men)
d["Sex_group"] = d[GEN_COL].astype(str).str.strip().replace({"0": "Women", "1": "Men"})

# Age_group like paper (>=65 only)
age_bins = [65, 75, 85, np.inf]
age_labels = ["[65–75[", "[75–85[", "≥85"]
d["Age_group"] = pd.cut(d[AGE_COL], bins=age_bins, right=False, labels=age_labels)

d = d.dropna(subset=["Sex_group","Age_group"]).copy()

# Merge reference (ref_table1 must exist)
needed = ["Sex_group","Age_group","w_mean","w_sd","h_mean","h_sd","bmi_mean","bmi_sd","hg_mean","hg_sd"]
missing = [c for c in needed if c not in ref_table1.columns]
if missing:
    raise ValueError(f"[COMPARE ERROR] ref_table1 missing columns: {missing}")

m = d.merge(ref_table1[needed], on=["Sex_group","Age_group"], how="left")

if m[["w_mean","h_mean","bmi_mean","hg_mean"]].isna().any(axis=1).any():
    bad = m[m[["w_mean","h_mean","bmi_mean","hg_mean"]].isna().any(axis=1)][["Sex_group","Age_group"]].drop_duplicates()
    raise ValueError(f"[COMPARE ERROR] Some strata not found in ref_table1:\n{bad}")

# 2) Tests
def one_sample_tests(z):
    z = pd.to_numeric(z, errors="coerce").dropna()
    if len(z) < 5:
        return {"n": len(z), "mean_z": np.nan, "p_ttest": np.nan, "median_z": np.nan, "p_wilcoxon": np.nan}
    t = ttest_1samp(z, 0.0)
    # Wilcoxon can fail if all zeros; handle safely
    try:
        w = wilcoxon(z, zero_method="wilcox")
        p_w = w.pvalue
    except Exception:
        p_w = np.nan
    return {
        "n": int(len(z)),
        "mean_z": float(z.mean()),
        "p_ttest": float(t.pvalue),
        "median_z": float(np.median(z)),
        "p_wilcoxon": float(p_w),
    }

def run_var(var_label, x_series, mu, sd):
    z = (x_series - mu) / sd
    out = []

    # All
    out.append({"variable": var_label, "group": "All", **one_sample_tests(z)})

    # Women / Men
    for sex in ["Women", "Men"]:
        idx = (m["Sex_group"] == sex)
        out.append({"variable": var_label, "group": sex, **one_sample_tests(z[idx])})

    return out

results = []
results += run_var("Weight (kg)",    m[W_COL],   m["w_mean"],   m["w_sd"])
results += run_var("Height (cm)",    m[H_COL],   m["h_mean"],   m["h_sd"])
results += run_var("BMI (kg/m²)",    m["BMI"],   m["bmi_mean"], m["bmi_sd"])
results += run_var("Handgrip (kgf)", m[HG_COL],  m["hg_mean"],  m["hg_sd"])

results = pd.DataFrame(results)

# nice formatting
def fmt_p(p):
    if pd.isna(p): return ""
    return f"{p:.3g}"

results["mean_z"]   = results["mean_z"].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
results["median_z"] = results["median_z"].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
results["p_ttest"]  = results["p_ttest"].map(fmt_p)
results["p_wilcoxon"] = results["p_wilcoxon"].map(fmt_p)

results[["variable","group","n","mean_z","p_ttest","median_z","p_wilcoxon"]]


,variable,group,n,mean_z,p_ttest,median_z,p_wilcoxon
0,Weight (kg),All,2457,-0.046,0.00726,-0.131,1.04e-06
1,Weight (kg),Women,1638,-0.053,0.00992,-0.141,1.21e-05
2,Weight (kg),Men,819,-0.032,0.298,-0.107,0.0258
3,Height (cm),All,2457,-0.045,0.0152,-0.085,0.0128
4,Height (cm),Women,1638,-0.016,0.478,0.017,0.604
5,Height (cm),Men,819,-0.101,0.000759,-0.132,0.00113
6,BMI (kg/m²),All,2457,-0.007,0.687,-0.095,0.00823
7,BMI (kg/m²),Women,1638,-0.030,0.159,-0.126,0.00299
8,BMI (kg/m²),Men,819,0.039,0.211,-0.013,0.766
9,Handgrip (kgf),All,2457,0.097,3.01e-05,0.196,2.95e-15


When compared against Portuguese reference values stratified by sex and age group, our cohort was slightly lighter and shorter overall, with no meaningful differences in BMI. Handgrip strength was higher than reference values, driven mainly by women, while men showed no relevant differences in strength.

In [ ]:
import numpy as np
import pandas as pd

# 0) Helpers: detetar colunas no df_final
def pick_col(df, preferred, keywords_all):
    if preferred in df.columns:
        return preferred
    cands = [c for c in df.columns if all(k.lower() in c.lower() for k in keywords_all)]
    return cands[0] if cands else None

def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def require_col(name, col, df):
    if col is None or col not in df.columns:
        raise ValueError(f"[TABLE ERROR] Missing column for '{name}'. Got: {col}")

# 1) Mapear colunas
GEN_COL  = "Genero" if "Genero" in df_final.columns else pick_col_any(df_final, ["Genero","Género","Sexo","Sex"], ["genero","género","sexo","sex"])
AGE_COL  = pick_col_any(df_final, ["Idade","Idade_anos","Age","IDADE"], ["idade","age"])

DIST_COL = pick_col(df_final, "6_MIN_ANDAR_Distancia_TOTAL_metros", ["6_MIN", "Distancia"])
TUG_COL  = pick_col(df_final, "Best_TIMED_UP_AND_GO_Simples", ["TIMED_UP_AND_GO", "Simples"])

# NOVO: Sit-to-Stand (reps)
STS_COL  = pick_col_any(
    df_final,
    ["SIT_TO_STAND_Repetições", "SIT_TO_STAND_Repeticoes", "SIT_TO_STAND_Repeticoes_total", "SIT_TO_STAND_Repeticoes_Total",
     "SIT_TO_STAND_Repeticoes", "SIT_TO_STAND_Repeticoes"],
    ["sit_to_stand", "repet", "stand"]
)

require_col("Genero", GEN_COL, df_final)
require_col("Idade", AGE_COL, df_final)
require_col("6-min walk distance", DIST_COL, df_final)
require_col("TUG", TUG_COL, df_final)
require_col("Sit-to-Stand reps", STS_COL, df_final)

# 2) Preparar dataset
df = df_final.copy()
df[AGE_COL]  = pd.to_numeric(df[AGE_COL], errors="coerce")
df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")
df[TUG_COL]  = pd.to_numeric(df[TUG_COL], errors="coerce")
df[STS_COL]  = pd.to_numeric(df[STS_COL], errors="coerce")

# Sexo para labels
df["Sex_group"] = df[GEN_COL].astype(str).str.strip().replace({"0": "Women", "1": "Men"})

# Intervalos etários como no exemplo (60–64, 65–69, 70–74, 75–79)
age_bins = [60, 65, 70, 75, 80]
age_labels = ["60–64", "65–69", "70–74", "75–79"]
df["Age_interval"] = pd.cut(df[AGE_COL], bins=age_bins, right=False, labels=age_labels)

# Mantém só idades dentro de 60–79 (para ficar igual ao exemplo)
df = df.dropna(subset=["Sex_group", "Age_interval"]).copy()

# 3) Funções para mean (SD)
def mean_sd(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return ""
    return f"{s.mean():.1f} ({s.std(ddof=1):.1f})"

def build_block(df, value_col, row_name):
    """Devolve DataFrame com linhas Men/Women e colunas por intervalo + Combined."""
    rows = []
    for sex in ["Men", "Women"]:
        d = df[df["Sex_group"] == sex].copy()
        r = {"Functional-fitness test": row_name}
        for lab in age_labels:
            r[lab] = mean_sd(d.loc[d["Age_interval"] == lab, value_col])
        r["Combined"] = mean_sd(d[value_col])
        r["Sex"] = sex
        rows.append(r)

    out = pd.DataFrame(rows).set_index(["Sex", "Functional-fitness test"])
    return out

# 4) Construir tabela (3 variáveis)
tbl_sts  = build_block(df, STS_COL,  "chair stand, n")  # Sit-to-Stand reps
tbl_dist = build_block(df, DIST_COL, "6-min walk, m")
tbl_tug  = build_block(df, TUG_COL,  "8-ft up-and-go, s")

table_like = pd.concat([tbl_sts, tbl_dist, tbl_tug], axis=0)

table_like



,,60–64,65–69,70–74,75–79,Combined
Sex,Functional-fitness test,,,,,
Men,"chair stand, n",16.3 (4.2),16.7 (4.1),15.9 (4.4),15.2 (3.8),16.0 (4.2)
Women,"chair stand, n",15.8 (4.1),15.4 (3.8),14.7 (4.0),13.9 (4.0),14.9 (4.0)
Men,"6-min walk, m",547.7 (107.2),551.5 (83.6),528.1 (88.5),494.3 (101.0),527.9 (94.7)
Women,"6-min walk, m",522.3 (84.9),507.9 (82.2),476.7 (94.5),452.5 (86.9),489.1 (90.7)
Men,"8-ft up-and-go, s",5.9 (1.3),5.9 (1.3),6.4 (1.5),6.9 (1.5),6.3 (1.5)
Women,"8-ft up-and-go, s",6.4 (1.5),6.6 (1.3),7.2 (1.8),7.9 (1.9),7.0 (1.7)


In [39]:
import pandas as pd

# ============================================================
# Reference table (from your screenshot) for:
# - chair stand, n   (Sit-to-Stand reps)
# - 6-min walk, m
# - 8-ft up-and-go, s
# By Sex_group x Age_interval
# ============================================================

ref_rows = [
    # ---------------- Men ----------------
    {"Sex_group":"Men", "Age_interval":"60–64",
     "chair_mean":15.7, "chair_sd":4.1,
     "sixmin_mean":577.9, "sixmin_sd":93.7,
     "tug_mean":4.8, "tug_sd":1.5},

    {"Sex_group":"Men", "Age_interval":"65–69",
     "chair_mean":14.8, "chair_sd":4.0,
     "sixmin_mean":526.9, "sixmin_sd":115.7,
     "tug_mean":5.4, "tug_sd":2.1},

    {"Sex_group":"Men", "Age_interval":"70–74",
     "chair_mean":13.4, "chair_sd":4.0,
     "sixmin_mean":512.3, "sixmin_sd":105.9,
     "tug_mean":5.9, "tug_sd":2.1},

    {"Sex_group":"Men", "Age_interval":"75–79",
     "chair_mean":12.6, "chair_sd":3.1,
     "sixmin_mean":461.8, "sixmin_sd":108.6,
     "tug_mean":6.9, "tug_sd":3.2},

    {"Sex_group":"Men", "Age_interval":"Combined",
     "chair_mean":14.2, "chair_sd":4.0,
     "sixmin_mean":521.2, "sixmin_sd":113.3,
     "tug_mean":5.7, "tug_sd":2.4},

    # ---------------- Women ----------------
    {"Sex_group":"Women", "Age_interval":"60–64",
     "chair_mean":14.8, "chair_sd":4.7,
     "sixmin_mean":502.6, "sixmin_sd":97.0,
     "tug_mean":5.3, "tug_sd":1.3},

    {"Sex_group":"Women", "Age_interval":"65–69",
     "chair_mean":13.2, "chair_sd":4.2,
     "sixmin_mean":474.8, "sixmin_sd":110.1,
     "tug_mean":6.0, "tug_sd":1.7},

    {"Sex_group":"Women", "Age_interval":"70–74",
     "chair_mean":12.8, "chair_sd":3.7,
     "sixmin_mean":452.7, "sixmin_sd":98.1,
     "tug_mean":6.5, "tug_sd":2.4},

    {"Sex_group":"Women", "Age_interval":"75–79",
     "chair_mean":11.5, "chair_sd":3.4,
     "sixmin_mean":392.8, "sixmin_sd":118.2,
     "tug_mean":7.7, "tug_sd":3.6},

    {"Sex_group":"Women", "Age_interval":"Combined",
     "chair_mean":13.2, "chair_sd":4.2,
     "sixmin_mean":457.5, "sixmin_sd":112.9,
     "tug_mean":6.3, "tug_sd":2.5},
]

ref_table_ff = pd.DataFrame(ref_rows)

ref_table_ff




,Sex_group,Age_interval,chair_mean,chair_sd,sixmin_mean,sixmin_sd,tug_mean,tug_sd
0,Men,60–64,15.7,4.1,577.9,93.7,4.8,1.5
1,Men,65–69,14.8,4.0,526.9,115.7,5.4,2.1
2,Men,70–74,13.4,4.0,512.3,105.9,5.9,2.1
3,Men,75–79,12.6,3.1,461.8,108.6,6.9,3.2
4,Men,Combined,14.2,4.0,521.2,113.3,5.7,2.4
5,Women,60–64,14.8,4.7,502.6,97.0,5.3,1.3
6,Women,65–69,13.2,4.2,474.8,110.1,6.0,1.7
7,Women,70–74,12.8,3.7,452.7,98.1,6.5,2.4
8,Women,75–79,11.5,3.4,392.8,118.2,7.7,3.6
9,Women,Combined,13.2,4.2,457.5,112.9,6.3,2.5


In [40]:
ref_table_ff_wide = (
    ref_table_ff
    .set_index(["Sex_group", "Age_interval"])
    [["chair_mean","chair_sd", "sixmin_mean","sixmin_sd", "tug_mean","tug_sd"]]
    .sort_index()
)

ref_table_ff_wide



chair_mean  chair_sd  sixmin_mean  sixmin_sd  \
Sex_group Age_interval                                                 
Men       60–64               15.7       4.1        577.9       93.7   
          65–69               14.8       4.0        526.9      115.7   
          70–74               13.4       4.0        512.3      105.9   
          75–79               12.6       3.1        461.8      108.6   
          Combined            14.2       4.0        521.2      113.3   
Women     60–64               14.8       4.7        502.6       97.0   
          65–69               13.2       4.2        474.8      110.1   
          70–74               12.8       3.7        452.7       98.1   
          75–79               11.5       3.4        392.8      118.2   
          Combined            13.2       4.2        457.5      112.9   

                        tug_mean  tug_sd  
Sex_group Age_interval                    
Men       60–64              4.8     1.5  
          65–69              5.4     2.1  
          70–74              5.9     2.1  
          75–79              6.9     3.2  
          Combined           5.7     2.4  
Women     60–64              5.3     1.3  
          65–69              6.0     1.7  
          70–74              6.5     2.4  
          75–79              7.7     3.6  
          Combined           6.3     2.5

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon

# 0) Checks: ref_table_ff tem de existir (inclui chair_* agora)
needed_ref = {
    "Sex_group","Age_interval",
    "chair_mean","chair_sd",
    "sixmin_mean","sixmin_sd",
    "tug_mean","tug_sd"
}
missing_ref = list(needed_ref - set(ref_table_ff.columns))
if missing_ref:
    raise ValueError(f"[COMPARE ERROR] ref_table_ff missing columns: {missing_ref}")

# usamos só estratos (sem Combined) para o merge pessoa-a-pessoa
ref_strata = ref_table_ff[ref_table_ff["Age_interval"].isin(["60–64","65–69","70–74","75–79"])].copy()

# 1) Detetar colunas no df_final
def pick_col(df, preferred, keywords_all):
    if preferred in df.columns:
        return preferred
    cands = [c for c in df.columns if all(k.lower() in c.lower() for k in keywords_all)]
    return cands[0] if cands else None

def pick_col_any(df, preferred_list, keywords_any):
    for p in preferred_list:
        if p in df.columns:
            return p
    keys = [k.lower() for k in keywords_any]
    cands = [c for c in df.columns if any(k in c.lower() for k in keys)]
    return cands[0] if cands else None

def require_col(name, col, df):
    if col is None or col not in df.columns:
        raise ValueError(f"[COMPARE ERROR] Missing column for '{name}'. Got: {col}")

GEN_COL  = "Genero" if "Genero" in df_final.columns else pick_col_any(df_final, ["Genero","Género","Sexo","Sex"], ["genero","género","sexo","sex"])
AGE_COL  = pick_col_any(df_final, ["Idade","Idade_anos","AGE","Age"], ["idade","age"])

DIST_COL = pick_col(df_final, "6_MIN_ANDAR_Distancia_TOTAL_metros", ["6_MIN", "Distancia"])
TUG_COL  = pick_col(df_final, "Best_TIMED_UP_AND_GO_Simples", ["TIMED_UP_AND_GO", "Simples"])

# NOVO: Sit-to-Stand reps
STS_COL = pick_col_any(
    df_final,
    ["SIT_TO_STAND_Repetições", "SIT_TO_STAND_Repeticoes", "SIT_TO_STAND_Repeticoes_total", "SIT_TO_STAND_Repeticoes_Total",
     "SIT_TO_STAND_Repeticoes"],
    ["sit_to_stand", "repet", "stand", "chair"]
)

for nm, col in [
    ("Genero",GEN_COL),
    ("Idade",AGE_COL),
    ("6-min distância",DIST_COL),
    ("TUG",TUG_COL),
    ("Sit-to-Stand reps",STS_COL),
]:
    require_col(nm, col, df_final)

# 2) Preparar dados + bins etários como na referência
df = df_final.copy()
df[AGE_COL]  = pd.to_numeric(df[AGE_COL], errors="coerce")
df[DIST_COL] = pd.to_numeric(df[DIST_COL], errors="coerce")
df[TUG_COL]  = pd.to_numeric(df[TUG_COL], errors="coerce")
df[STS_COL]  = pd.to_numeric(df[STS_COL], errors="coerce")

df["Sex_group"] = df[GEN_COL].astype(str).str.strip().replace({"0":"Women", "1":"Men"})

age_bins   = [60, 65, 70, 75, 80]
age_labels = ["60–64", "65–69", "70–74", "75–79"]
df["Age_interval"] = pd.cut(df[AGE_COL], bins=age_bins, right=False, labels=age_labels)

# mantém apenas quem entra nos estratos da referência
df = df.dropna(subset=["Sex_group","Age_interval"]).copy()

# 3) Merge com referência por Sex_group + Age_interval
m = df.merge(ref_strata, on=["Sex_group","Age_interval"], how="left")

need_merge = ["chair_mean","chair_sd","sixmin_mean","sixmin_sd","tug_mean","tug_sd"]
if m[need_merge].isna().any(axis=1).any():
    bad = m[m[need_merge].isna().any(axis=1)][["Sex_group","Age_interval"]].drop_duplicates()
    raise ValueError(f"[COMPARE ERROR] Some strata not found in ref_table_ff:\n{bad}")

# 4) Testes (t-test + Wilcoxon) sobre z-scores
def one_sample_tests(z):
    z = pd.to_numeric(z, errors="coerce").dropna()
    if len(z) < 5:
        return {"n": int(len(z)), "mean_z": np.nan, "p_ttest": np.nan, "median_z": np.nan, "p_wilcoxon": np.nan}
    t = ttest_1samp(z, 0.0)
    try:
        w = wilcoxon(z, zero_method="wilcox")
        p_w = w.pvalue
    except Exception:
        p_w = np.nan
    return {
        "n": int(len(z)),
        "mean_z": float(z.mean()),
        "p_ttest": float(t.pvalue),
        "median_z": float(np.median(z)),
        "p_wilcoxon": float(p_w),
    }

def run_var(var_label, x, mu, sd):
    z = (x - mu) / sd
    out = []
    out.append({"variable": var_label, "group": "All", **one_sample_tests(z)})
    for sex in ["Women","Men"]:
        idx = (m["Sex_group"] == sex)
        out.append({"variable": var_label, "group": sex, **one_sample_tests(z[idx])})
    return out

results = []
results += run_var("chair stand (n)", m[STS_COL],  m["chair_mean"],  m["chair_sd"])
results += run_var("6-min walk (m)",  m[DIST_COL], m["sixmin_mean"], m["sixmin_sd"])
results += run_var("8-ft up-and-go / TUG (s)", m[TUG_COL], m["tug_mean"], m["tug_sd"])

results = pd.DataFrame(results)

# formatação
def fmt_p(p):
    if pd.isna(p): return ""
    return f"{p:.3g}"

results["mean_z"] = results["mean_z"].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
results["median_z"] = results["median_z"].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
results["p_ttest"] = results["p_ttest"].map(fmt_p)
results["p_wilcoxon"] = results["p_wilcoxon"].map(fmt_p)

results[["variable","group","n","mean_z","p_ttest","median_z","p_wilcoxon"]]




,variable,group,n,mean_z,p_ttest,median_z,p_wilcoxon
0,chair stand (n),All,2509,0.542,6.95e-130,0.452,1.99e-122
1,chair stand (n),Women,1710,0.517,5.72e-87,0.441,5.9e-82
2,chair stand (n),Men,799,0.595,6.67e-45,0.452,1.18e-41
3,6-min walk (m),All,2509,0.262,8.04e-51,0.352,1.16e-74
4,6-min walk (m),Women,1710,0.308,1.33e-48,0.411,3.48e-64
5,6-min walk (m),Men,799,0.163,1.75e-07,0.260,3.96e-14
6,8-ft up-and-go / TUG (s),All,2509,0.305,8.34e-81,0.129,1.06e-59
7,8-ft up-and-go / TUG (s),Women,1710,0.342,2.12e-62,0.169,1.47e-49
8,8-ft up-and-go / TUG (s),Men,799,0.225,2.7e-20,0.081,2.04e-12


When benchmarked against sex- and age-matched normative values, our cohort showed clearly better performance in the chair-stand test and the 6-minute walk test, indicating higher lower-limb strength and greater functional endurance, consistent with a generally more active and fitter population. In contrast, TUG times were longer than the reference; however, this result should be interpreted with caution because the TUG is highly sensitive to test configuration and procedural details (e.g., walking distance and course layout, chair height, turning instructions, allowance of assistive devices, footwear, and timing method). Therefore, the observed discrepancy is likely influenced—at least in part—by methodological differences rather than true inferior mobility. Overall, the pattern suggests that our sample may be healthier and more physically active than the reference population, which can introduce selection bias and should be acknowledged when interpreting comparisons and generalizing results.